# Demo: cosa distingue gli schemi di uno studente da quelli generati da un'AI

Gira tutto qui, **senza chiavi, credenziali o Google Drive**: solo codice (niente modelli, niente OCR), dati di esempio inclusi nel repo (schemi dello studente + pagine generate da Claude; nessuna pagina del docente).

Esegui le celle in ordine (*Runtime > Esegui tutto*). Circa 2-3 minuti in tutto.

## 1. Scarica il repo e installa

In [ ]:
!git clone -q https://github.com/03gilbe-design/pacchetto-schemi-studio.git
%cd pacchetto-schemi-studio/strumenti/3_numeri_da_appunti
!pip install -q -r requirements.txt
!python -m playwright install --with-deps chromium > /dev/null 2>&1 && echo chromium pronto

## 2. Trova i numeri che distinguono lo studente dall'AI
Confronta 7 pagine di appunti dello studente (PDF) con 7 pagine generate da Claude (HTML/SVG). Scrive in `risultati_numeri/` la classifica, le frasi da dare al modello e i range per il controllo.

In [ ]:
!python -X utf8 trova_differenze.py --suoi esempi_numeri/suoi/Architettura_Gilberto.pdf --generate esempi_numeri/generate --prof-parole 385

from IPython.display import Image, Markdown, display
display(Image('risultati_numeri/classifica.png'))
display(Markdown(open('risultati_numeri/FRASI_PER_IL_MODELLO.md', encoding='utf-8').read()))

## 3. Controlla una pagina generata: OK o FUORI
Rende in PNG una pagina di esempio (Claude con le regole) e la confronta con i range dello studente. Ogni FUORI ha il consiglio per correggerla.

In [ ]:
from pathlib import Path
from trova_differenze import rendi
src = Path('esempi_numeri/generate/claude_CON_REGOLE_r1/pagina.html')
rendi(src, Path('pagina.png'))
display(Image('pagina.png', width=400))
!python -X utf8 controlla_stile.py pagina.png --sorgente {src} --rif risultati_numeri/numeri.json --prof-parole 385

## 4. (Opzionale) Con i tuoi file
Carica **un PDF dei tuoi schemi** e **una o piu' pagine generate da un'AI** (`.html`, `.svg` o `.png`). I file restano in questa sessione Colab.

In [ ]:
from google.colab import files
import shutil, os
print('1) i tuoi schemi (PDF o PNG)'); suoi = list(files.upload())
print('2) le pagine generate (.html/.svg/.png)'); gen = list(files.upload())
os.makedirs('miei/generate', exist_ok=True)
for f in suoi: shutil.move(f, 'miei/' + f)
for f in gen: shutil.move(f, 'miei/generate/' + f)
args = ' '.join(f'"miei/{f}"' for f in suoi)
!python -X utf8 trova_differenze.py --suoi {args} --generate miei/generate --out risultati_miei
display(Image('risultati_miei/classifica.png'))
display(Markdown(open('risultati_miei/FRASI_PER_IL_MODELLO.md', encoding='utf-8').read()))